In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## படி 1: கட்டமைக்கப்பட்ட வெளியீடுகளுக்கான Pydantic மாதிரிகளை வரையறு

இந்த மாதிரிகள் முகவர்கள் திருப்பி விடும் **திட்டத்தை** வரையறுக்கின்றன. Pydantic உடன் `response_format` ஐ பயன்படுத்துவது உறுதி செய்கிறது:
- ✅ வகை-பாதுகாப்பான தரவு மூலம் எடுப்பு
- ✅ தானாக சரிபார்ப்பு
- ✅ விலக்கு உரை பதில்களிலிருந்து எந்த தவறும் இல்லாமை
- ✅ புலங்களின் அடிப்படையில் எளிய நிபந்தனை வழித்தடம்


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## படி 2: ஹோட்டல் முன்பதிவு கருவியை உருவாக்குங்கள்

இந்த கருவி **availability_agent** அழைக்கும் இடமாகும், இது அறைகள் கிடைக்குமா என்று சரிபார்க்கும். நாங்கள் `@ai_function` அலங்காரியை பயன்படுத்துகிறோம்:
- ஒரு Python செயல்பாட்டை AI அழைக்கக்கூடிய கருவியாக மாற்ற
- LLM க்கான JSON திட்டத்தை தானாக உருவாக்க
- அளவுரு சரிபார்ப்பை கையாள
- முகவர்கள் மூலம் தானாக அழைக்க இயலும்

இந்த காட்சி நிகழ்ச்சிக்கான:
- **ஸ்டாக்ஹோம், ஸீடில், டோக்கியோ, லண்டன், ஆம்ஸ்டர்டாம்** → அறைகள் ✅
- **மற்ற அனைத்து நகரங்களும்** → அறைகள் இல்லை ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## படி 3: வழிசெலுத்தின் நிலைமைக் கூற்றுக்களை வரையறு

இந்தக் கூற்றுகள் முகவர் பதிலை ஆய்வு செய்து, வேலைதிட்டத்தில் எந்த பாதையை எடுத்துப்போக வேண்டும் என்பதை தீர்மானிக்கின்றன.

**முக்கிய வடிவமைப்பு:**
1. செய்தி `AgentExecutorResponse` ஆக உள்ளதா என பார்
2. கட்டமைக்கப்பட்ட வெளியீட்டை (Pydantic மாடல்) பகுப்பாய்வு செய்
3. வழிச் செலுத்தலை கட்டுப்படுத்த `True` அல்லது `False` ஆக திரும்பி வா

வேலைதிட்டம், அடுத்த எய்க்ஸிகியூடரை அழைக்க எந்தவெளியீடுகளை பயன்படுத்த வேண்டும் என **கூறுகளின் விளிம்புகளில்** இந்தக் கூற்றுக்களை மதிப்பீடு செய்யும்.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## படி 4: தனிப்பயன் காட்சி இயக்கியை உருவாக்குக

இயக்கிகள் மாற்றங்கள் அல்லது புறவிளைவுகள் செய்யும் பணியாளர்கள். நாம் இறுதி விளைவினை காட்சிபடுத்தும் தனிப்பயன் இயக்கியை உருவாக்க `@executor` அலங்காரியை பயன்படுத்துகிறோம்.

**முக்கிய கருத்துக்கள்:**
- `@executor(id="...")` - ஒரு செயல்வழி இயக்கியாக ஒரு பணியை பதிவு செய்கிறது
- `WorkflowContext[Never, str]` - உள்ளீடு/வெளிப்படும் வகை குறிகாட்டிகள்
- `ctx.yield_output(...)` - இறுதி செயல்வழி விளைவினை வழங்குகிறது


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## படி 5: சுற்றுச்சூழல் மாறிகள் ஏற்றவும்

LLM கிளையன்டை காட்சிப்படுத்தவும். இந்த எடுத்துக்காட்டு பின்வருவனுடன் வேலை செய்கிறது:
- **GitHub மாதிரிகள்** (GitHub டோக்கன் கொண்ட இலவச மட்டம்)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## படி 6: கட்டமைக்கப்பட்ட வெளியீடுகளுடன் AI முகவரிகளை உருவாக்குங்கள்

நாம் **மூன்று சிறப்பான முகவரிகளை** உருவாக்குகிறோம், ஒவ்வொன்றும் `AgentExecutor` இல் இயங்குகிறது:

1. **availability_agent** - கருவியைப் பயன்படுத்தி ஹோட்டல் கிடைப்பைச் சோதிக்கிறது
2. **alternative_agent** - மாற்று நகரங்களை பரிந்துரைக்கிறது (அறைகள் இல்லாதபோது)
3. **booking_agent** - முன்பதிவு செய்வதற்கும் ஊக்குவிக்கிறது (அறைகள் கிடைக்கும் போது)

**முக்கிய அம்சங்கள்:**
- `tools=[hotel_booking]` - முகவரிக்கு கருவியை வழங்குகிறது
- `response_format=PydanticModel` - கட்டமைக்கப்பட்ட JSON வெளியீட்டை கட்டாயப்படுத்துகிறது
- `AgentExecutor(..., id="...")` - வேலைப்பாட்டிற்கு முகவரியை சுற்றி வைக்கிறது


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## படி 7: நிபந்தனை அங்குகளுடன் வேலைதிட்டத்தை கட்டமைக்கவும்

இப்போது நிபந்தனை வழிமுறையுடன் கிராஃப்பை கட்டமைக்க `WorkflowBuilder` ஐ நாங்கள் பயன்படுத்துகிறோம்:

**வேலைதிட்ட அமைப்பு:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**முக்கிய முறைமைகள்:**
- `.set_start_executor(...)` - நுழைவு புள்ளியை அமைக்கிறது
- `.add_edge(from, to, condition=...)` - நிபந்தனை அங்கை சேர்க்கிறது
- `.build()` - வேலைதிட்டத்தை இறுதி நிலையில் மாற்றுகிறது


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## படி 8: சோதனை வழக்கு 1 இயக்கவும் - உள்ளமைவு இல்லாத நகரம் (பாரீஸ்)

எங்கள் சிமுலேஷனில் அறைகள் இல்லை என்ற பாரீஸ் நகரத்தில் ஹோட்டல்களை கேட்க **உள்ளமைவு இல்லாத** வழியை சோதிக்கலாம்.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## படி 9: சோதனை விவரம் 2 - கிடைக்கும் இடத்துடன் நகரம் (ஸ்டாக்ஹோம்)

இப்போது நாம் **கிடைக்கும்தன்மை** பாதையை சோதிக்க ஸ்டாக்ஹோம் (எமது சிமுலேஷனில் அறைகள் உள்ள நகரம்) உள்ள ஹோட்டல்களை கேட்கிறோம்.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## முக்கியமான கையாள்வுகள் மற்றும் அடுத்த படிகள்

### ✅ நீங்கள் கற்றுக்கொண்டவை:

1. **WorkflowBuilder மாதிரி**
   - நுழைவு புள்ளியை நிர்ணயிக்க `.set_start_executor()` பயன்படுத்தவும்
   - நிபந்தனை சார்ந்த வழிமாற்றுக்கு `.add_edge(from, to, condition=...)` பயன்படுத்தவும்
   - Workflow ஐ இறுதிப்படுத்த `.build()` அழைக்கவும்

2. **நிபந்தனை சார்ந்த வழிமாற்று**
   - நிலைமைகள் `AgentExecutorResponse` ஐ பரிசீலிக்கின்றன
   - கட்டமைக்கப்பட்ட வெளியீடுகளை பகுப்பாய்வு செய்து வழிமாற்று முடிவுகள் எடுக்கின்றன
   - ஒரு பக்கம் செயல்படுத்த `True` திருப்பவும், தவிர்க்க `False` திருப்பவும்

3. **கருவி ஒருங்கிணைப்பு**
   - Python functions ஐ AI கருவிகளாக மாற்ற `@ai_function` பயன்படுத்தவும்
   - முகவர்கள் தேவைப்படும்போது கருவிகளை தானாக அழைக்கின்றனர்
   - கருவிகள் முகவர்கள் பகுப்பாய்வு செய்யக்கூடிய JSON மறுமொழி அளிக்கின்றன

4. **கட்டமைக்கப்பட்ட வெளியீடுகள்**
   - வகை-பாதுகாப்பான தரவுகளை எடுக்கும் Pydantic மாதிரிகளை பயன்படுத்தவும்
   - முகவர்கள் உருவாக்கும் போது `response_format=MyModel` நிறுவவும்
   - பதில்களை `Model.model_validate_json()` மூலம் பகுப்பாய்வு செய்யவும்

5. **தனிப்பயன் இயக்கிகள்**
   - Workflow கூறுகளை உருவாக்க `@executor(id="...")` பயன்படுத்தவும்
   - இயக்கிகள் தரவை மாற்றவோ அல்லது பக்க விளைவுகளை செய்யவோ முடியும்
   - Workflow முடிவுகளை உழைக்கும் `ctx.yield_output()` பயன்படுத்தவும்

### 🚀 உண்மையான உலக பயன்பாடுகள்:

- **பயண முன்பதிவு**: கிடைக்கும் நிலையை சரிபார், மாற்று விருப்பங்களை பரிந்துரைக்கவும், ஒப்பிடுக
- **வாடிக்கையாளர் சேவை**: பிரச்சனை வகை, உணர்ச்சி, முன்னுரிமை அடிப்படையில் வழிமாற்று செய்யவும்
- **மின்வணிகம்**: பட்டியலை சரிபார், மாற்று பரிந்துரைகள், ஆணைகளை செயலாக்கு
- **உள்ளடக்க பராமரிப்பு**: விஷம மதிப்பெண்கள், பயனர் கொடுப்பனவுகள் அடிப்படையில் வழிமாற்று
- **அங்கீகாரம் வேலைப்பாட்டு**: தொகை, பயனர் வேட்புக்கள், அபாய நிலை அடிப்படையில் வழிமாற்று
- **பன்முக செயலாக்கம்**: தர தரம், முழுமை அடிப்படையில் வழிமாற்று செய்யவும்

### 📚 அடுத்த படிகள்:

- கூடுதல் சிக்கலான நிபந்தனைகள் சேர்க்கவும் (பல அளவுருக்கள்)
- Workflow நிலை மேலாண்மையுடன் முறுக்களை நடைமுறைப்படுத்தவும்
- மீண்டும் பயன்படுத்தக்கூடிய கூறுகளுக்கான துணை-workflows சேர்க்கவும்
- உண்மையான API களுடன் ஒருங்கிணைக்கவும் (ஹோட்டல் முன்பதிவு, பட்டியல் அமைப்புகள்)
- பிழை கையாளல் மற்றும் மாற்று வழிமுறைகள் சேர்க்கவும்
- உள்ளமைந்த காட்சிப்படுத்தல் கருவிகளுடன் workflows களை காட்சிப்படுத்தவும்


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
